# 数据层验证
验证类型往返、真实提交与跨连接读取、异常回滚。测试结束仅删除本次生成的UUID记录。

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from data import Database, MySQLConfig
from data import codec
database = Database(MySQLConfig.load())
database.health()

{'database_name': 'FRM', 'server_version': '9.7.2', 'schema_version': 1}

In [2]:
frame = pd.DataFrame({
    'return': [0.01, np.nan, -0.03],
    'count': pd.Series([1, None, 3], dtype='Int64'),
    'label': pd.Series(['甲', None, '乙'], dtype='string'),
    'group': pd.Categorical(['a', 'b', 'a'], ordered=True),
    'date': pd.date_range('2026-01-01', periods=3),
})
pd.testing.assert_frame_equal(frame, codec.loads(codec.dumps(frame)))
indexed = frame.set_index('date')
pd.testing.assert_frame_equal(indexed, codec.loads(codec.dumps(indexed)))
np.testing.assert_equal(codec.loads(codec.dumps(np.array([np.inf, -np.inf, np.nan]))), [np.inf, -np.inf, np.nan])
print('类型往返通过')

类型往返通过


In [3]:
dataset_id = None
try:
    with database.session() as repository:
        dataset_id = repository.save_dataset('integration-test', frame, metadata={'unit': 'decimal'})
    with database.session() as repository:
        snapshot = repository.load_dataset(dataset_id)
        pd.testing.assert_frame_equal(frame, snapshot.frame)
        assert snapshot.metadata == {'unit': 'decimal'}
    print('真实提交及跨连接读取通过')
finally:
    if dataset_id:
        with database.session() as repository:
            with repository.connection.cursor() as cursor:
                cursor.execute('DELETE FROM frm_datasets WHERE dataset_id=%s', (dataset_id,))

真实提交及跨连接读取通过


In [4]:
try:
    with database.session() as repository:
        rolled_back = repository.save_dataset('rollback-test', frame)
        raise RuntimeError('deliberate test failure')
except RuntimeError:
    pass
with database.session() as repository:
    try:
        repository.load_dataset(rolled_back)
    except KeyError:
        print('异常事务回滚通过')
    else:
        raise AssertionError('Transaction did not roll back')

异常事务回滚通过
